# RecVis Assignment 3 — Baselines

Train on labeled CIFAR-10 and submit predictions for the unlabeled diffusion-hard test set. **Validation accuracy is measured on ordinary CIFAR-10 and is not expected to predict Kaggle accuracy.**

Attach the competition data through Kaggle's **Add Input** panel and enable a GPU. On Kaggle, update the default input path if needed. For local execution, set the `RECVIS_INPUT_DIR` and optional `RECVIS_OUTPUT_DIR` environment variables.

In [ ]:
# Kaggle already includes PyTorch. Install this only for the optional CLIP cell.
# !pip install -q open-clip-torch

from pathlib import Path
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

# The environment variables make the same notebook testable outside Kaggle.
INPUT_DIR = Path(os.environ.get('RECVIS_INPUT_DIR', '/kaggle/input/recvis-mva-a3-2026'))
OUTPUT_DIR = Path(os.environ.get('RECVIS_OUTPUT_DIR', '/kaggle/working'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
CLASS_NAMES = ('airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
required_files = {'train.npz', 'validation.npz', 'test.npz', 'sample_submission.csv'}
missing_files = sorted(name for name in required_files if not (INPUT_DIR / name).is_file())
if missing_files:
    raise FileNotFoundError(f'Missing competition files in {INPUT_DIR}: {missing_files}')
print(f'Using {DEVICE}; input: {INPUT_DIR}')
print(sorted(path.name for path in INPUT_DIR.iterdir()))

## Load the data

`train.npz` and `validation.npz` are labeled CIFAR-10. `test.npz` contains only image arrays and identifiers: it intentionally has no labels.

In [ ]:
class NpzDataset(Dataset):
    def __init__(self, path, transform, labeled):
        with np.load(path) as archive:
            self.images = archive['images'].astype(np.uint8)
            self.labels = archive['labels'].astype(np.int64) if 'labels' in archive else None
            self.ids = archive['ids'].astype(np.int64) if 'ids' in archive else np.arange(len(self.images))
        if labeled and self.labels is None:
            raise ValueError(f'{path} must have labels')
        self.transform, self.labeled = transform, labeled
    def __len__(self): return len(self.images)
    def __getitem__(self, index):
        image = self.transform(Image.fromarray(self.images[index], mode='RGB'))
        return image, int(self.labels[index] if self.labeled else self.ids[index])

CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
def transforms_for(image_size=32, pretrained=False):
    mean, std = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) if pretrained else (CIFAR_MEAN, CIFAR_STD)
    if image_size == 32:
        train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean, std)])
        eval_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    else:
        train_tf = T.Compose([T.RandomResizedCrop(image_size, scale=(0.8, 1.0)), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean, std)])
        eval_tf = T.Compose([T.Resize((image_size, image_size)), T.ToTensor(), T.Normalize(mean, std)])
    return train_tf, eval_tf

def loaders(image_size=32, pretrained=False, batch_size=128, workers=2):
    train_tf, eval_tf = transforms_for(image_size, pretrained)
    train_set = NpzDataset(INPUT_DIR / 'train.npz', train_tf, labeled=True)
    val_set = NpzDataset(INPUT_DIR / 'validation.npz', eval_tf, labeled=True)
    options = dict(batch_size=batch_size, num_workers=workers, pin_memory=DEVICE.type == 'cuda')
    return DataLoader(train_set, shuffle=True, **options), DataLoader(val_set, shuffle=False, **options)

with np.load(INPUT_DIR / 'test.npz') as test_archive:
    print('test keys:', test_archive.files, '| test images:', test_archive['images'].shape)
    assert 'labels' not in test_archive.files, 'Test labels must never be public.'

## Supervised baseline: ResNet-18

Start with this fast, native-resolution baseline. Increase `EPOCHS` for a serious run. Try ImageNet pretraining and 224px inputs in the next section.

In [ ]:
MODEL_NAME = 'resnet18'  # resnet18, resnet50, or vit_b_16
PRETRAINED = False
LINEAR_PROBE = False  # requires PRETRAINED=True
IMAGE_SIZE = 32
BATCH_SIZE = 128
EPOCHS = 5  # starter setting; increase for final experiments

def build_model(name, pretrained, image_size):
    if name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        if image_size == 32 and not pretrained:
            model.conv1, model.maxpool = nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.Identity()
        model.fc = nn.Linear(model.fc.in_features, 10)
    elif name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        model.fc = nn.Linear(model.fc.in_features, 10)
    elif name == 'vit_b_16':
        if image_size != 224: raise ValueError('ViT-B/16 baseline requires IMAGE_SIZE = 224')
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None)
        model.heads.head = nn.Linear(model.heads.head.in_features, 10)
    else: raise ValueError(name)
    return model

train_loader, val_loader = loaders(IMAGE_SIZE, PRETRAINED, BATCH_SIZE)
model = build_model(MODEL_NAME, PRETRAINED, IMAGE_SIZE).to(DEVICE)
if LINEAR_PROBE:
    if not PRETRAINED:
        raise ValueError('LINEAR_PROBE=True requires PRETRAINED=True')
    for parameter in model.parameters():
        parameter.requires_grad = False
    head = model.fc if hasattr(model, 'fc') else model.heads
    for parameter in head.parameters():
        parameter.requires_grad = True
optimizer = torch.optim.AdamW(
    (parameter for parameter in model.parameters() if parameter.requires_grad),
    lr=3e-4,
    weight_decay=1e-4,
)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler(DEVICE.type, enabled=DEVICE.type == 'cuda')

def run_epoch(loader, training):
    model.train(training); total_loss = total_correct = total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training), autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(images); loss = criterion(logits, labels)
        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total += len(labels); total_loss += loss.item() * len(labels)
        total_correct += (logits.argmax(1) == labels).sum().item()
    return total_loss / total, total_correct / total

best_state, best_accuracy, history = None, -1, []
for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader, training=False)
    history.append({'epoch': epoch, 'train_accuracy': train_accuracy, 'val_accuracy': val_accuracy})
    print(f'{epoch:02d}: train={train_accuracy:.4f}, CIFAR-10 validation={val_accuracy:.4f}')
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
pd.DataFrame(history)

### Transfer-learning variants

For an ImageNet-pretrained ResNet use `MODEL_NAME='resnet18'`, `PRETRAINED=True`, and `IMAGE_SIZE=224`. To perform linear probing, freeze all parameters except `model.fc` before creating the optimizer. For ViT, use `MODEL_NAME='vit_b_16'`, `PRETRAINED=True`, and `IMAGE_SIZE=224`.

All external checkpoints and data must be disclosed in the report.

## Generate a Kaggle submission

This uses the held-out test images without attempting to load labels. Upload the resulting `/kaggle/working/submission.csv` file.

In [ ]:
@torch.inference_mode()
def make_submission(model, image_size, pretrained, output_path):
    _, test_tf = transforms_for(image_size, pretrained)
    test_set = NpzDataset(INPUT_DIR / 'test.npz', test_tf, labeled=False)
    test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2, pin_memory=DEVICE.type == 'cuda')
    model.eval(); ids, predictions = [], []
    for images, batch_ids in test_loader:
        predictions.extend(model(images.to(DEVICE)).argmax(dim=1).cpu().tolist())
        ids.extend(batch_ids.tolist())
    submission = pd.DataFrame({'id': ids, 'label': predictions})
    if list(submission.columns) != ['id', 'label']:
        raise ValueError('Submission columns must be exactly: id,label')
    if len(submission) != len(test_set) or submission['id'].duplicated().any():
        raise ValueError('Submission must contain each test id exactly once')
    if not np.array_equal(submission['id'].to_numpy(), test_set.ids):
        raise ValueError('Submission ids are not aligned with test.npz')
    if submission['label'].isna().any() or not submission['label'].between(0, 9).all():
        raise ValueError('Predicted labels must be integers from 0 to 9')
    sample = pd.read_csv(INPUT_DIR / 'sample_submission.csv')
    if sample.columns.tolist() != ['id', 'label'] or not np.array_equal(sample['id'].to_numpy(), test_set.ids):
        raise ValueError('sample_submission.csv is not aligned with test.npz')
    submission.to_csv(output_path, index=False)
    return submission

submission = make_submission(model, IMAGE_SIZE, PRETRAINED, OUTPUT_DIR / 'submission.csv')
submission.head(), submission.shape

## Optional: open-source VLMs

You may use open-source vision-language models such as OpenCLIP or Qwen-VL-family models, subject to the assignment policy. Proprietary models and API-only services are prohibited. Record model/checkpoint names, prompts, external data, and all tuning choices in the report.

A zero-shot OpenCLIP baseline can be implemented by encoding one prompt per class and assigning the class with highest image/text similarity. Consider prompt ensembling, but validate carefully: source-domain validation and the hidden target domain differ.